In [ ]:
import glob
import os
import sqlite3
import pandas as pd
import numpy as np
import networkx as nx
from collections import Counter
import matplotlib.pyplot as plt
from tqdm.notebook import trange, tqdm
import itertools
from collections import defaultdict

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import rcParams
import seaborn as sns

plt.style.use('ggplot')

rcParams['font.family'] = 'sans-serif'
rcParams['font.style'] = 'normal'

rcParams['figure.facecolor'] = 'white'
rcParams['figure.dpi'] = 150

rcParams['savefig.bbox'] = 'tight'
rcParams['savefig.dpi'] = 300
rcParams['savefig.transparent'] = True

rcParams['axes.spines.right'] = False
rcParams['axes.spines.top'] = False
rcParams['axes.labelsize'] = 10
rcParams['axes.labelcolor'] = 'black'
rcParams['axes.edgecolor'] = 'black'
rcParams['axes.linewidth'] = 1
rcParams['axes.facecolor'] = 'white'

rcParams['legend.fontsize'] = 6

rcParams['xtick.color'] = 'black'
rcParams['ytick.color'] = 'black'
rcParams['xtick.major.width'] = 1
rcParams['ytick.major.width'] = 1
rcParams['xtick.major.size'] = 2
rcParams['ytick.major.size'] = 2
rcParams['xtick.labelsize'] = 8
rcParams['ytick.labelsize'] = 8

rcParams['lines.linewidth'] = 1
rcParams['lines.markersize'] = 5

rcParams['grid.color'] = 'white'
rcParams['grid.linewidth'] = 0.0

# Define settings and runs

In [ ]:
networks = ['BA', 'ER']
recsyss = ['F', 'FP', 'P', 'RC']
runs = list(range(10))  # 0-9

# Average number of recommendations

In [ ]:
# data_for_plots[(network,recsys)][round] = [counts across runs]
data_for_plots = defaultdict(lambda: defaultdict(list))

pbar = tqdm(
    total=len(networks)*len(recsyss)*len(runs),
    desc="Processing Experiments",
    unit="experiment"
)

for network in networks:
    for recsys in recsyss:

        key = (network, recsys)

        for run in runs:

            db_path = f"experiments_recsys/{network}_{recsys}_{run}/database_server.db"

            with sqlite3.connect(db_path) as conn:
                cursor = conn.cursor()

                cursor.execute("""
                    SELECT round, COUNT(*)
                    FROM recommendations
                    GROUP BY round
                """)

                # fetch aggregated results only
                for r, count in cursor.fetchall():
                    data_for_plots[key][r].append(count)

            pbar.update(1)

        # compute stats AFTER all runs for this (network,recsys)
        rounds = list(data_for_plots[key].keys())

        avg = {r: np.mean(data_for_plots[key][r]) for r in rounds}
        std = {r: np.std(data_for_plots[key][r]) for r in rounds}

        data_for_plots[key]['avg'] = avg
        data_for_plots[key]['std'] = std

pbar.close()


In [ ]:
fig, axes = plt.subplots(2, sharey=True, figsize=(12, 8))

for i, network in enumerate(networks):
    for recsys in recsyss:
        ax = axes[i]
        rounds = np.array(list(data_for_plots[(network, recsys)]['avg'].keys()))
        avg_counts = np.array(list(data_for_plots[(network, recsys)]['avg'].values()))
        std_counts = np.array(list(data_for_plots[(network, recsys)]['std'].values()))
        
        cum_counts = np.cumsum(avg_counts)
        
        ax.plot(rounds, cum_counts, label=f'{recsys} (Avg)')
        ax.fill_between(rounds, cum_counts - std_counts, cum_counts + std_counts, alpha=0.2)
        ax.set_xlabel('Round')
        ax.set_ylabel('Average Number of Recommendations')
        ax.legend()
plt.show()

del data_for_plots  # free memory

# How many times individual posts are recommended?

In [ ]:
data_for_plots = {}

pbar = tqdm(
    total=len(networks)*len(recsyss)*len(runs),
    desc="Processing Experiments",
    unit="experiment"
)

for network in networks:
    for recsys in recsyss:

        key = (network, recsys)

        # store histogram per run
        run_hists = []

        for run in runs:

            post_counter = Counter()

            db_path = f"experiments_recsys/{network}_{recsys}_{run}/database_server.db"

            with sqlite3.connect(db_path) as conn:
                cursor = conn.cursor()
                cursor.execute("SELECT post_ids FROM recommendations")

                for (post_ids,) in cursor:
                    for pid in post_ids.split("|"):
                        post_counter[int(pid)] += 1

            # histogram: how many posts got k recs
            hist = Counter(post_counter.values())
            run_hists.append(hist)

            pbar.update(1)

        # ---- average histogram across runs ----

        all_k = set().union(*run_hists)

        avg_hist = {}
        std_hist = {}

        for k in all_k:
            vals = [h.get(k, 0) for h in run_hists]
            avg_hist[k] = np.mean(vals)
            std_hist[k] = np.std(vals)

        data_for_plots[key] = {
            "avg": avg_hist,
            "std": std_hist,
            "raw": run_hists
        }

pbar.close()
